# Часть F, веха M0. Окружения, пакет urec и вопросы TOFU

Проверяем в Colab то, что сделано в вехе M0: оба окружения собираются из lock-файлов репозитория, пакет `urec` ставится в `atk`, не меняя его, самопроверка `scripts/check_env.sh` печатает OK, а команда `prepare-data` на Linux даёт ровно те файлы `data/items`, что лежат в репозитории. Нужна любая GPU Colab (хватит T4); всё вместе — около 20 минут. Перед этой частью должны быть выполнены части A–E.
Подробный разбор — в файле [docs/F_M0_explained.md](https://github.com/IvanovskyDev/Machine-Unlearning-in-LLM/blob/main/docs/F_M0_explained.md).

**1. Старт сессии** — как шаг 1 части E.

In [ ]:
import os                                   # папки и переменные окружения

from google.colab import drive, userdata    # Google Drive и секреты Colab

drive.mount("/content/drive")               # подключить Google Drive

DRIVE = "/content/drive/MyDrive/unlearning_data"   # папка проекта на Drive (постоянная)
FAST = "/content/fast"                             # папка на диске машины (очищается после сессии)
REPO = "/content/repo"                             # сюда шаг 3 клонирует репозиторий диплома

os.makedirs(DRIVE + "/saves", exist_ok=True)       # результаты OpenUnlearning
os.makedirs(DRIVE + "/results_raw", exist_ok=True) # сырые результаты атак
os.makedirs(DRIVE + "/logs", exist_ok=True)        # логи запусков и замеры времени и памяти
os.makedirs(FAST + "/hf_home", exist_ok=True)      # кэш Hugging Face
os.makedirs(FAST + "/models", exist_ok=True)       # скачанные модели

os.environ["BIG"] = DRIVE                          # «большой диск» из плана
os.environ["REPO"] = REPO                          # путь к репозиторию для ячеек %%bash
os.environ["HF_HOME"] = FAST + "/hf_home"          # кэш Hugging Face
os.environ["MODELS"] = FAST + "/models"            # папка моделей
os.environ["TOKENIZERS_PARALLELISM"] = "false"     # меньше лишних предупреждений
os.environ["PYTHONUNBUFFERED"] = "1"               # вывод программ сразу попадает в лог
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # токен Hugging Face из секрета HF_TOKEN

print("Старт сессии выполнен")

**2. Ставим uv и убираем настройки Colab, которые мешают окружениям** — как шаг 2 части E.

In [ ]:
# Colab задаёт свои переменные окружения, которые мешают нашим окружениям:
#   UV_...      — велят uv ставить пакеты в системный Python 3.13 Colab с его ограничениями версий;
#   PYTHONPATH  — подмешивает модули Colab в любой запущенный Python;
#   MPLBACKEND  — настройка графиков блокнота; в наших окружениях из-за неё падают vLLM и BERTScore.
for name in list(os.environ):
    if name.startswith("UV_") or name in ["PYTHONPATH", "MPLBACKEND"]:
        print("Убираю", name, "=", os.environ.pop(name))

# поставить uv (-q — без подробного вывода) и проверить, что он работает
!pip install -q uv
!uv --version

**3. Клонируем репозиторий диплома вместе с форком** — как шаг 3 части E. Должны напечататься коммит репозитория, коммит форка и пять файлов `data/items`.

In [ ]:
%%bash
set -e                                     # остановиться на первой ошибке
rm -rf $REPO                               # удалить старую копию, если есть (данные на Drive не трогаются)
git clone -q --recurse-submodules https://github.com/IvanovskyDev/Machine-Unlearning-in-LLM.git $REPO
cd $REPO
git log -1 --format='репозиторий: %h %s'  # последний коммит репозитория диплома
git submodule status                      # какой коммит форка закреплён в репозитории
ls -l data/items                          # вопросы TOFU, которые записала команда prepare-data

**4. Собираем окружение `unl` из lock-файла репозитория** — как шаг 6 части E. Несколько минут; проверим его в шаге 6.

In [ ]:
%%bash
set -e
# unl: новое окружение, пакеты ровно по lock-файлу, затем FlashAttention (его в lock-файле нет)
uv venv /content/envs/unl --python 3.11 --seed --clear
source /content/envs/unl/bin/activate
uv pip install -r $REPO/envs/requirements-unl.lock
uv pip install "https://github.com/Dao-AILab/flash-attention/releases/download/v2.6.3/flash_attn-2.6.3+cu123torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl"
echo "Окружение unl готово"

**5. Собираем окружение `atk` и ставим в него пакет `urec`** (блок 25). Несколько минут. Установка `urec` должна закончиться строками `Installed 1 package` и `+ urec==0.1.0`: всё, что нужно пакету, в `atk` уже есть, и окружение не меняется.

In [ ]:
%%bash
set -e
# atk: новое окружение, пакеты ровно по lock-файлу (как шаг 14 части B, но файл — из репозитория)
uv venv /content/envs/atk --python 3.11 --seed --clear
source /content/envs/atk/bin/activate
uv pip install -r $REPO/envs/requirements-atk.lock
# -e — «режим разработки»: в окружение ставится ссылка на папку src/urec, а не копия кода
uv pip install -e $REPO
python -c "import urec; print('urec из папки', urec.__path__[0])"

**6. Самопроверка окружений** (блок 9): `scripts/check_env.sh` проверяет оба окружения и печатает OK или FAIL по каждой проверке. Около трёх минут: в первый раз BERTScore скачивает модель roberta-large (1,4 ГБ). Вывод сохраняется на Drive в `logs/check_env.log`. В конце должно быть `OK: все проверки прошли`.

In [ ]:
%%bash
set -o pipefail                            # ошибка скрипта не теряется в конвейере с tee
# tee — показать вывод и одновременно записать его в файл на Drive
bash $REPO/scripts/check_env.sh 2>&1 | tee $BIG/logs/check_env.log

**7. Готовим вопросы TOFU и сверяем с репозиторием** (блок 25). Команда `prepare-data` пишет файлы в отдельную папку, а `cmp` сравнивает их побайтно с файлами репозитория. Для каждого из пяти файлов должно напечататься «совпадает».

In [ ]:
%%bash
set -e
CHECK=/content/fast/prepare_check          # отдельная папка: файлы репозитория не трогаем
rm -rf $CHECK
# paths.data=… — добавка к конфигу: писать не в data/ репозитория, а в $CHECK
/content/envs/atk/bin/python -m urec.cli prepare-data paths.data=$CHECK
for name in forget01 forget05 forget10 retain_eval retain_pool; do
  cmp $CHECK/items/$name.jsonl $REPO/data/items/$name.jsonl && echo "$name.jsonl: совпадает с репозиторием"
done

**8. Смотрим на вопросы** — сколько их в каждом файле, имена авторов forget05 и одна пара «вопрос — ответ».

In [ ]:
%%bash
cd $REPO
# python - <<'EOF' … EOF — выполнить в окружении atk программу, записанную прямо здесь
/content/envs/atk/bin/python - <<'EOF'
from urec.io import read_jsonl
from urec.types import QAItem

for name in ["forget01", "forget05", "forget10", "retain_eval", "retain_pool"]:
    items = read_jsonl(f"data/items/{name}.jsonl", QAItem)
    print(f"{name}: {len(items)} вопросов")

forget05 = read_jsonl("data/items/forget05.jsonl", QAItem)
print()
print("Авторы forget05:")
for item in forget05[::20]:                 # [::20] — каждая 20-я запись: первая пара каждого автора
    print("  ", item.author_local, item.author)

item = forget05[0]
print()
print(item.item_id, "|", item.author)
print("Вопрос:", item.question)
print("Ответ: ", item.answer)
EOF

**9. Записываем веху M0 в журнал** (`journal.md` на Drive). Итог самопроверки берётся из `logs/check_env.log`, сверка файлов повторяется. Строку «Наблюдения» допишите сами.

In [ ]:
import filecmp                              # побайтное сравнение файлов
from datetime import datetime
from zoneinfo import ZoneInfo

# коммиты репозитория диплома и форка; -C папка — выполнить git в этой папке
repo_commit = !git -C $REPO log -1 --format=%h
fork_commit = !git -C $REPO/external/open-unlearning log -1 --format=%h
gpu = !nvidia-smi --query-gpu=name,driver_version --format=csv,noheader
today = datetime.now(ZoneInfo("Europe/Moscow")).date()

with open(DRIVE + "/logs/check_env.log", encoding="utf-8") as f:
    check_env = f.read().strip().splitlines()[-1]   # последняя строка: «OK: …» или «FAIL: …»

names = ["forget01", "forget05", "forget10", "retain_eval", "retain_pool"]
same = [
    filecmp.cmp(f"{REPO}/data/items/{name}.jsonl", f"{FAST}/prepare_check/items/{name}.jsonl", shallow=False)
    for name in names
]
items_result = "совпадают с репозиторием" if all(same) else "НЕ совпадают с репозиторием"

text = f"""
## {today} — Часть F, веха M0: окружения, пакет urec, вопросы TOFU (Colab)
- Где: Google Colab, {gpu[0]} (имя GPU, драйвер)
- Репозиторий: Machine-Unlearning-in-LLM {repo_commit[0]}; форк open-unlearning {fork_commit[0]}
- Окружения unl и atk — из envs/ репозитория; пакет urec установлен в atk
- scripts/check_env.sh: {check_env}
- prepare-data: файлы data/items {items_result}
- Наблюдения: …
"""

with open(DRIVE + "/journal.md", "a", encoding="utf-8") as f:   # дописать запись в конец журнала
    f.write(text)

print(text)

**10. Завершаем сессию**: дожидаемся, пока все файлы запишутся на Drive, и отключаем его. Чтобы продолжить работу после этого шага, начните снова с шага 1.

In [ ]:
from google.colab import drive

drive.flush_and_unmount()          # записать на Drive всё, что ещё не записано, и отключить его
print("Все файлы записаны на Drive. Машину можно отключить: Runtime → Disconnect and delete runtime")